In [1]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import UnexpectedAlertPresentException, NoAlertPresentException
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
# 1. 설정
TARGET_URL = "https://www.krit.re.kr/krit/bbs/gbgs_list.do?gotoMenuNo=03090100"
SAVE_DIR = os.path.abspath("국기연_발간물")

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

chrome_options = Options()
prefs = {
    "download.default_directory": SAVE_DIR,
    "download.prompt_for_download": False,
    "directory_upgrade": True,
    "safebrowsing.enabled": False,
    "plugins.always_open_pdf_externally": True  # PDF가 브라우저에서 열리지 않고 바로 다운로드되게 함
}
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument('--ignore-certificate-errors')
chrome_options.add_argument('--ignore-ssl-errors')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

def start_download():
    try:
        driver.get(TARGET_URL)
        wait = WebDriverWait(driver, 15)
        
        # 1. 리스트 로딩 대기 (이미지의 ul.imgList type2 구조)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul.imgList.type2 li")))
        
        # 2. 모든 게시글(li) 요소 찾기
        items = driver.find_elements(By.CSS_SELECTOR, "ul.imgList.type2 > li")
        print(f"발견된 게시글 개수: {len(items)}개")

        for i in range(len(items)): 
            try:
                # 페이지 이동이나 DOM 변화에 대비해 매번 요소를 새로 잡습니다.
                current_items = driver.find_elements(By.CSS_SELECTOR, "ul.imgList.type2 > li")
                item = current_items[i]
                
                # 제목 추출 (이미지의 span 태그 내용)
                try:
                    title_text = item.find_element(By.CSS_SELECTOR, "div span").text
                except:
                    title_text = f"Unknown_Title_{i}"
                
                print(f"[{i+1}/{len(items)}] 처리 중: {title_text}")

                # 3. 다운로드 버튼 찾기 (href에 download.do가 포함된 a 태그)
                # 이미지 구조상 '바로보기'와 '다운로드' 링크가 따로 있을 수 있으므로 필터링합니다.
                download_links = item.find_elements(By.CSS_SELECTOR, "a[href*='download.do']")
                
                if not download_links:
                    print(f"   -> 다운로드 링크를 찾을 수 없습니다.")
                    continue

                for link in download_links:
                    try:
                        file_name = link.get_attribute("title") or title_text
                        print(f"   -> 다운로드 실행: {file_name}")
                        
                        # 클릭 시 자바스크립트 충돌 방지를 위해 execute_script 사용
                        driver.execute_script("arguments[0].click();", link)
                        
                        # 다운로드 완료를 위한 충분한 대기 (파일 크기에 따라 조절 필요)
                        time.sleep(3) 

                        # 알림창 처리
                        try:
                            alert = driver.switch_to.alert
                            print(f"   ⚠️ 서버 알림: {alert.text}")
                            alert.accept()
                        except NoAlertPresentException:
                            pass

                    except Exception as e:
                        print(f"   ❌ 개별 파일 다운로드 중 오류: {e}")

            except Exception as e:
                print(f"[{i+1}번 항목] 처리 중 에러 발생: {e}")
                continue

        print(f"\n✅ 작업 완료! 파일 확인: {SAVE_DIR}")

    finally:
        # 다운로드가 완료될 때까지 잠시 기다렸다가 종료
        time.sleep(5)
        driver.quit()

if __name__ == "__main__":
    start_download()

발견된 게시글 개수: 8개
[1/8] 처리 중: '25~'39 국방기술기획서(일반본)
   -> 다운로드 실행: '25~'39 국방기술기획서(일반본)
[2/8] 처리 중: '24~'38 국방기술기획서(일반본)
   -> 다운로드 실행: '24~'38 국방기술기획서(일반본)
[3/8] 처리 중: '23-'37 국방기술기획서
   -> 다운로드 실행: '23-'37 국방기술기획서
[4/8] 처리 중: '22~'36 국방기술기획서(일반본)
   -> 다운로드 실행: '22~'36 국방기술기획서(일반본)
[5/8] 처리 중: '21~'35 핵심기술기획서(일반본)
   -> 다운로드 실행: '21~'35 핵심기술기획서(일반본)
[6/8] 처리 중: ’19~’33 핵심기술기획서
   -> 다운로드 실행: ’19~’33 핵심기술기획서
[7/8] 처리 중: ’18~’32 핵심기술기획서
   -> 다운로드 실행: ’18~’32 핵심기술기획서
[8/8] 처리 중: '17~'31 핵심기술기획서 일반본
   -> 다운로드 실행: '17~'31 핵심기술기획서 일반본

✅ 작업 완료! 파일 확인: c:\Users\user\OneDrive\Desktop\Desktop\VSCode\크롤링프로젝트\국기연_발간물
